In [1]:
import ast
import os
import pandas as pd

def extract_genus_name(x):
    try:
        genus_dict = ast.literal_eval(x)
        return genus_dict.get('name', None)
    except:
        return None

os.chdir('/active-data/analysis_results/chr_pla')
all_data = pd.read_csv('genome_chr-pla_statistics.csv')
all_data['genus_clean'] = all_data['genus'].apply(extract_genus_name)
counts = all_data['genus_clean'].value_counts()
keep_genus = counts[counts >= 400].index.to_list()
print(keep_genus)

['Escherichia', 'Klebsiella', 'Staphylococcus', 'Pseudomonas', 'Bacillus', 'Salmonella', 'Streptococcus', 'Streptomyces', 'Acinetobacter', 'Enterococcus', 'Bordetella', 'Enterobacter', 'Xanthomonas', 'Campylobacter', 'Vibrio', 'Mycobacterium', 'Corynebacterium', 'Burkholderia', 'Listeria', 'Citrobacter', 'Helicobacter']


In [2]:
from Bio import SeqIO
import pandas as pd

def seq_frac_calcu(align_result, lenth, acc_n, seq_id, pla_acc):
    temp_id_list = list(align_result['pident'].value_counts().index)
    add_counts_a = align_result['qstart'].value_counts()
    minor_counts_a = align_result['qend'].value_counts()
    
    align_result = align_result[align_result['sseqid'].isin(pla_acc)]
    add_counts_p = align_result['qstart'].value_counts()
    minor_counts_p = align_result['qend'].value_counts()
    
    add_num_a = 0
    add_num_p = 0
    fra_list = []
    for j in range(lenth):
        if j+1 in add_counts_a.index:
            add_num_a += add_counts_a[j+1]
        all_count = int(add_num_a)
        if j+1 in minor_counts_a.index:
            add_num_a -= minor_counts_a[j+1]
            
        if j+1 in add_counts_p.index:
            add_num_p += add_counts_p[j+1]
        pla_count = int(add_num_p)
        if j+1 in minor_counts_p.index:
            add_num_p -= minor_counts_p[j+1]
            
        try:
            fra = pla_count/all_count
        except:
            if acc_n+'-'+seq_id in pla_acc:
                fra = 1
            else:
                fra = 0
        fra_list.append(fra)
    return fra_list

def upper_cover_region(fra_list):
    check_points = [0.05, 0.1, 0.15, 0.2, 0.25, 0.5, 0.75, 0.8, 0.85, 0.9, 0.95]
    check_labels = {}
    cover_regions = {}
    temp_regions = {}
    for point in check_points:
        check_labels[point] = False
        cover_regions[point] = []
        temp_regions[point] = []
    site = 0
    for fra in fra_list:
        site += 1
        for point in check_points:
            if check_labels[point] == False and fra >= point:
                temp_regions[point].append(site)
                check_labels[point] = True
            elif check_labels[point] == True and fra < point:
                temp_regions[point].append(site-1)
                check_labels[point] = False
                cover_regions[point].append(temp_regions[point])
                temp_regions[point] = []
    for point in check_points:
        if check_labels[point] == True:
            temp_regions[point].append(site)
            check_labels[point] = False
            cover_regions[point].append(temp_regions[point])
            temp_regions[point] = []
    return cover_regions

def rec_by_len(fra_list, rec_len):
    frac_list = []
    sub_skip = False
    for j in range(int(len(fra_list)/rec_len) + 1):
        if j == int(len(fra_list)/rec_len) - 1 and len(fra_list[(j+1)*rec_len:]) < rec_len/2:
            sub_skip = True
            temp_fra_list = fra_list[j*rec_len:]
        else:
            temp_fra_list = fra_list[j*rec_len:(j+1)*rec_len]

        if sub_skip and j == int(len(fra_list)/rec_len):
            continue
        try:
            temp_ave_frac = sum(temp_fra_list)/len(temp_fra_list)
            frac_list.append(temp_ave_frac)
        except:
            continue
    return frac_list
    
def pla_frac_calcu(acc_n, genus_name, pla_acc, que):
    folder = f'/active-data/analysis_results/chr_pla/genus/cor-pla_fraction_records/{genus_name}/{acc_n}'
    if not os.path.exists(folder):
        os.makedirs(folder)
    file_path = f'{folder}/sum_replicon_bitscore.csv'
    if os.path.exists(file_path):
        pass
    else:
        handle = open(f'/active-data/genomes/bacteria_complete_annotationRefSeq-20250807/data/{acc_n}/genomic.gbff')
        acc_record = SeqIO.parse(handle, 'genbank')
        scores = {'original':{}, 'pident_90':{}, 'pident_95':{}}
        for seq_record in acc_record:
            tot_len = len(seq_record)
            rec_len = 200
            frac_list = {'original':[], 'pident_90':[], 'pident_95':[]}
            tot_count = {'original':0, 'pident_90':0, 'pident_95':0}
            upper_cover = {}
            
            blast_seq = seq_record
            os.chdir('/active-data/temp/blastn')
            temp_ncl_file = open(f'temp_nucleotide_seq_{acc_n}.fasta', 'w+')
            SeqIO.write(blast_seq, temp_ncl_file, "fasta")
            temp_ncl_file.close()
            os.system(f'blastn -query temp_nucleotide_seq_{acc_n}.fasta -db /active-data/analysis_results/chr_pla/genus/blast_data/{genus_name}/nucleotide_seq.blastdb -out blastn_results_{acc_n}.txt -evalue 1e-50 -max_target_seqs 100000 -max_hsps 3000 -outfmt 6 -num_threads 8')
            head = ['qseqid', 'sseqid', 'pident', 'length', 'mismatch', 'gapopen', 'qstart', 'qend', 'sstart', 'send', 'evalue', 'bitscore']
            align_result = pd.read_csv(f'blastn_results_{acc_n}.txt', sep = '\t|;', engine = 'python', header = None, names = head)
            for dir_label in frac_list:
                if '90' in dir_label:
                    fra_list = seq_frac_calcu(align_result[align_result['pident'] >= 90], len(blast_seq), acc_n, seq_record.id, pla_acc)
                elif '95' in dir_label:
                    fra_list = seq_frac_calcu(align_result[align_result['pident'] >= 95], len(blast_seq), acc_n, seq_record.id, pla_acc)
                else:
                    fra_list = seq_frac_calcu(align_result, len(blast_seq), acc_n, seq_record.id, pla_acc)
                frac_list[dir_label] += rec_by_len(fra_list, rec_len)
                tot_count[dir_label] += sum(fra_list)
                upper_cover[dir_label] = upper_cover_region(fra_list)
            for dir_label in scores:
                if '90' in dir_label:
                    score_result = align_result[align_result['pident'] >= 90].groupby(['sseqid'])['bitscore'].sum()
                    forward_score = align_result[(align_result['pident'] >= 90) & (align_result['sstart'] < align_result['send'])].groupby(['sseqid'])['bitscore'].sum()
                    reverse_score = align_result[(align_result['pident'] >= 90) & (align_result['sstart'] > align_result['send'])].groupby(['sseqid'])['bitscore'].sum()
                elif '95' in dir_label:
                    score_result = align_result[align_result['pident'] >= 95].groupby(['sseqid'])['bitscore'].sum()
                    forward_score = align_result[(align_result['pident'] >= 95) & (align_result['sstart'] < align_result['send'])].groupby(['sseqid'])['bitscore'].sum()
                    reverse_score = align_result[(align_result['pident'] >= 95) & (align_result['sstart'] > align_result['send'])].groupby(['sseqid'])['bitscore'].sum()
                else:
                    score_result = align_result.groupby(['sseqid'])['bitscore'].sum()
                    forward_score = align_result[align_result['sstart'] < align_result['send']].groupby(['sseqid'])['bitscore'].sum()
                    reverse_score = align_result[align_result['sstart'] > align_result['send']].groupby(['sseqid'])['bitscore'].sum()
                scores[dir_label][seq_record.id] = score_result
                scores[dir_label][seq_record.id + '_f'] = forward_score
                scores[dir_label][seq_record.id + '_r'] = reverse_score
                
            os.system(f'rm temp_nucleotide_seq_{acc_n}.fasta')
            os.system(f'rm blastn_results_{acc_n}.txt')
            del align_result
                
            os.chdir(folder)
            frac_file = open(f"{seq_record.id}.txt","w+")
            for dir_label in frac_list:
                frac_file.write(str({f'average plasmid fraction-{dir_label}': tot_count[dir_label]/len(seq_record), 
                                     f'fraction list-{dir_label}': frac_list[dir_label], 
                                     f'upper_cover_regions-{dir_label}': upper_cover[dir_label]
                                    }) + '\n')
            frac_file.close()
    
        rep_score = pd.DataFrame()
        sum_score = pd.DataFrame()
        for dir_label in scores:
            os.chdir(folder)
            rep_data = pd.DataFrame()
            acc_sum_data = forward_sum = reverse_sum = pd.Series()
            new_column = []
            for item in scores[dir_label]:
                rep_data = pd.concat([rep_data, scores[dir_label][item]], axis=1,)
                if '_f' in item:
                    forward_sum = forward_sum.add(scores[dir_label][item], fill_value=0)
                elif '_r' in item:
                    reverse_sum = reverse_sum.add(scores[dir_label][item], fill_value=0)
                else:
                    acc_sum_data = acc_sum_data.add(scores[dir_label][item], fill_value=0)
                new_column.append(item+'-'+dir_label)
            rep_data.columns = new_column
            rep_data.index.rename(None, inplace=True)
            rep_score = pd.concat([rep_score, rep_data], axis=1,)
    
            for sum_data, d_label in zip([acc_sum_data, forward_sum, reverse_sum], ['', '_f', '_r']):
                sum_data = pd.DataFrame(sum_data).reset_index()
                sum_data = pd.concat([sum_data['sseqid'].str.split('-', expand=True), sum_data], axis=1)
                sum_data.columns = ['acc_n', 'rec_id', 'sseqid', 'bitscore']
                sum_data = sum_data.groupby(['acc_n'])['bitscore'].sum()
                sum_data = pd.DataFrame(sum_data)
                sum_data.columns = [acc_n+d_label+'-'+dir_label]
                sum_data.index.rename(None, inplace=True)
                sum_score = pd.concat([sum_score, sum_data], axis=1,)
            
        rep_score.to_csv(f'replicon_bitscore.csv')
        sum_score.to_csv(f'sum_replicon_bitscore.csv')
        
        handle.close()
    que.put(1)

In [3]:
from tqdm import tqdm
import multiprocessing

def run_blast(org_data_n, genus_name, pla_acc):
    manager = multiprocessing.Manager()
    que = manager.Queue()
    
    par = 10
    tot = len(org_data_n)
    pool = multiprocessing.Pool(par)
    
    for acc_n in org_data_n['accession']:
        pool.apply_async(pla_frac_calcu, (acc_n, genus_name, pla_acc, que))
        
    pool.close()
    
    count = 0
    with tqdm(total = tot, desc=f'{genus_name}({tot})', leave=True, ncols=100, unit='B', unit_scale=True) as pbar:
        while True:
            if not que.empty():
                value = que.get(True)
                count += 1
                pbar.update(1)
                if count == tot:
                    break
            else:
                continue
    
    pool.join()

In [4]:
for genus_name in keep_genus:
    org_data_n = all_data[all_data['genus_clean'].str.contains(genus_name, na=False)].reset_index(drop=True)
    pla_acc = []
    for i in org_data_n.index:
        acc_n = org_data_n['accession'][i]
        pla_data = ast.literal_eval(org_data_n['plasmid contigs'][i])
        for item in pla_data:
            pla_acc.append(acc_n + '-' + item)

    run_blast(org_data_n, genus_name, pla_acc)

Escherichia(4204): 100%|█████████████████████████████████████| 4.20k/4.20k [49:29:51<00:00, 42.4s/B]
Klebsiella(3554): 100%|██████████████████████████████████████| 3.55k/3.55k [65:46:38<00:00, 66.6s/B]
Staphylococcus(2423): 100%|██████████████████████████████████| 2.42k/2.42k [20:35:20<00:00, 23.6s/B]
Pseudomonas(2343): 100%|█████████████████████████████████████| 2.34k/2.34k [37:47:33<00:00, 58.1s/B]
Bacillus(1976): 100%|████████████████████████████████████████| 1.98k/1.98k [13:00:42<00:00, 18.2s/B]
Salmonella(1853): 100%|██████████████████████████████████████| 1.85k/1.85k [11:51:38<00:00, 23.0s/B]
Streptococcus(1599): 100%|████████████████████████████████████| 1.60k/1.60k [3:55:14<00:00, 8.83s/B]
Streptomyces(1359): 100%|████████████████████████████████████| 1.36k/1.36k [30:08:16<00:00, 79.8s/B]
Acinetobacter(1234): 100%|████████████████████████████████████| 1.23k/1.23k [6:32:56<00:00, 19.1s/B]
Helicobacter(416): 100%|████████████████████████████████████████████| 416/416 [34:42<00:00,